In [ ]:
import pde_descriptions,system_prompt
from parse_agent.prompt import parse_prompt
from code_agent.prompt import code_prompt
from debug_agent.prompt import debug_prompt

from dotenv import load_dotenv
import json
from ollama import chat
from verify_script.verify_result import *

advection_description = pde_descriptions.advection_description
pde_description = advection_description.format(beta = 0.1)
#system_prompt = system_prompt.system_prompt
user_prompt = parse_prompt.format(
    user_input=pde_description,
)

response = chat(
    model='qwen3:8b',
    messages=[
        #{'role': 'system', 'content': system_prompt},po
        {'role': 'user', 'content': user_prompt},
    ],
)
print(response.message.content)

#deleat ```json at beginning of response.text if exist
response_text = response.message.content
if response_text.startswith("```json"):
    response_text = response_text[len("```json"):]

# #deleat ``` at end of response.text if exist
if response_text.endswith("```"):
    response_text = response_text[:-3]

with open("./result/parsed_resp.json", "w", encoding="utf-8") as f:
    f.write(response_text)

# read json
with open("./result/parsed_resp.json", "r") as f:
    parsed_resp = json.load(f)
print(parsed_resp.keys())

{
  "PDEs": "\\frac{\\partial u}{\\partial t} = \\beta \\frac{\\partial u}{\\partial x}",
  "number_of_state_variables": 1,
  "texture_size": 1024,
  "spatial_step": 0.0009765625,
  "domain_size": 1.0,
  "temporal_step": 0.001,
  "time_horizon": 200.0,
  "boundary_conditions": "Periodic",
  "parameter_values": {"beta": 0.1},
  "notes": null
}
dict_keys(['PDEs', 'number_of_state_variables', 'texture_size', 'spatial_step', 'domain_size', 'temporal_step', 'time_horizon', 'boundary_conditions', 'parameter_values', 'notes'])


In [4]:
number_of_state_variables = parsed_resp["number_of_state_variables"]
texture_size = parsed_resp["texture_size"]
spatial_step = parsed_resp["spatial_step"]
temporal_step = parsed_resp["temporal_step"]
time_horizon = parsed_resp["time_horizon"]
#initial_condition_file = '"C:\\Users\\xan37\\OneDrive - Georgia Institute of Technology\\Documents\\GitHub\\cardiac-agent\\baselines models\\fk_data\\tau_d_0.5714\\IC.csv"'
parameter_values = parsed_resp["parameter_values"]
parameter_values
parameter_str = "\n".join([
    f"    const float {k:} = {v};" 
    for k, v in parameter_values.items()
])

coding_skeleton_file = f"./skeleton_script/{parsed_resp['number_of_state_variables']}V/skeleton.html"
# Load the file
with open(coding_skeleton_file, 'r') as f:
    html_template = f.read()

# Replace placeholders with your Python variables
updated_html = html_template.replace('{{DT_VALUE}}', str(temporal_step)).replace('{{DX_VALUE}}', str(spatial_step))
updated_html = updated_html.replace('{{TEXTURE_VALUE}}', str(texture_size))
# Optional: Save it back to a new file
with open('./result/skeleton.html', 'w') as f:
    f.write(updated_html)
    

In [5]:
with open(f"./skeleton_script/{parsed_resp['number_of_state_variables']}V/march_skeleton.frag", 'r') as f:
    coding_skeleton = f.read()
updated_coding_skeleton = coding_skeleton.replace('{{PARAMETER_VALUES}}', parameter_str)
user_prompt = code_prompt.format(
    PDEs=parsed_resp["PDEs"],
    coding_skeleton = updated_coding_skeleton
)

In [9]:
response = chat(
    #model = 'gemma4:31b',
    model='qwen3:8b',
    messages=[
        #{'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt},
    ],
)
print(response.message.content)


precision highp float;
precision highp int;

uniform sampler2D inTexture;
uniform float dt, dx, dy;

in vec2 cc, pixPos;

layout (location = 0) out vec4 ocolor;

#define u color.r
const float beta = 0.1;

// your codes here for helper function

// your codes here for helper function

// Main body of the shader
void main() {
    vec2 size = vec2(textureSize(inTexture, 0)) ;

    vec2 ii = vec2(1.,0.)/size ;
    vec2 jj = vec2(0.,1.)/size ;

    // read the color of the pixel
    vec4 color = texture( inTexture , cc ) ;


    // your codes here (do not define helper function)
    vec2 nextX = cc + ii;
    vec4 nextColor = texture(inTexture, nextX);
    float u_next = nextColor.r;
    float du_dx = (u_next - u) / dx;
    float du_dt = beta * du_dx;
    float u_new = u + dt * du_dt;
    ocolor.r = u_new;
    // your codes here (do not define helper function)

    // ocolor = color ;
    return ;
}


In [13]:
#deleat ```glsl at beginning of response.text if exist
response_text = response.message.content
if response_text.startswith("```glsl"):
    response_text = response_text[len("```glsl"):]

# #deleat ``` at end of response.text if exist
if response_text.endswith("```"):
    response_text = response_text[:-3]  

with open("./result/march_shader.frag", "w", encoding="utf-8") as f:
    f.write(response_text)


In [14]:
# replace march shader script in skeleton.html with the generated march shader code
with open('./result/skeleton.html', 'r') as f:
    html_content = f.read()
with open('./result/march_shader.frag', 'r') as f:    march_shader_code = f.read()
updated_html = html_content.replace('{{MARCH_SHADER_CODE}}', march_shader_code)
# save to new html
with open('./result/updated_skeleton.html', 'w') as f:
    f.write(updated_html)

## verify result

In [ ]:
IC_file = "./data/pdebench/random_sample_t0.csv"
simulation_file = './result/updated_skeleton.html'


In [12]:
import debug_agent
import glob,os

log_files = glob.glob("*.log")
for log_file in log_files:
    with open(log_file, 'r', encoding='utf-8') as f:
        original_log_content = f.read()

debug_prompt = debug_agent.debug_prompt.format(
    shader_codes = response.message.content,
    log_info = original_log_content
)

debug_response = chat(
    model='qwen2.5-coder:3b',
    messages=[
        #{'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': debug_prompt},
    ],
)
print(debug_response.message.content)

Based on the error messages you provided, it seems there are syntax errors in your GLSL code. Specifically, there's an issue with the use of the `.` operator in the shader code, which is causing a compilation error. Let's clean up the shader to ensure all syntax is correct and properly linked.

Here's the corrected version of the shader:

```glsl
precision highp float ;
precision highp int ;

uniform sampler2D   inTexture ;
uniform float       dt, D, dx, dy;

in vec2 cc, pixPos ;

layout (location = 0) out vec4 ocolor ;

#define u       color.r
#define v       color.g
#define w       color.b

const float tau_pv = 7.99;
const float tau_v1 = 9.8;
const float tau_v2 = 312.5;
const float tau_pw = 870.0;
const float tau_mw = 41.0;
const float tau_d  = 0.5714;
const float tau_0  = 12.5;
const float tau_r  = 33.83;
const float tau_si = 29.0;
const float K      = 10.0;
const float V_csi  = 0.861;
const float V_c = 0.13; 
const float V_v = 0.04;
const float C_m    = 1.0;

// Function to calcula

In [13]:
#deleat ```glsl at beginning of response.text if exist
response_text = debug_response.message.content
if response_text.startswith("```glsl"):
    response_text = response_text[len("```glsl"):]

# #deleat ``` at end of response.text if exist
if response_text.endswith("```"):
    response_text = response_text[:-3]

with open("march_shader.frag", "w", encoding="utf-8") as f:
    f.write(response_text)